# Schema variables: a collections policy knob
Collections only calls customers whose oldest open item is **N days past due**. Today N is hard-coded in procs and dashboards. Here it becomes a **schema variable**: a typed, persistent value scoped to `SAP_FINANCIALS`. Finance can change it with one `SET`, and every view and app that reads it picks up the new value. It needs no warehouse to read or write and no object redefinition.

**Before you run this:** the view below reads `COLLECTIONS_WORKLIST_DAILY`, which `CALL SP_BUILD_WORKLIST(25)` fills (`sql/03_procs.sql`). Build the environment first (see the repo README). The last cell sets the policy back to 30 days, so the demo can be run again.

In [ ]:
-- Demo hygiene: make sure every SELECT below re-executes instead of reusing a cached result
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

CREATE VARIABLE IF NOT EXISTS OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE NUMBER DEFAULT 30
  COMMENT = 'Collections policy: customers enter the call list once their oldest item is this many days past due';
SET OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE = 30;


In [ ]:
%%sql -r dataframe_1
SET OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE = 30;

In [ ]:
%%sql -r show_vars
SHOW SCHEMA VARIABLES IN SCHEMA OFFICEHOURS_20261027.SAP_FINANCIALS;

## Use it in a view
View bodies must use the **qualified** form `$db.schema.name`.

In [ ]:
%%sql -r create_view
CREATE OR REPLACE VIEW OFFICEHOURS_20261027.SAP_FINANCIALS.V_CALL_LIST AS
SELECT w.*
FROM OFFICEHOURS_20261027.SAP_FINANCIALS.COLLECTIONS_WORKLIST_DAILY w
WHERE w.RUN_DATE = (SELECT MAX(RUN_DATE) FROM OFFICEHOURS_20261027.SAP_FINANCIALS.COLLECTIONS_WORKLIST_DAILY)
  AND w.OLDEST_DAYS_PAST_DUE >= $OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE;

In [ ]:
%%sql -r call_list_30
SELECT $OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE AS CUTOFF_DAYS,
       COUNT(*) AS CUSTOMERS, ROUND(SUM(ACTIONABLE_USD)) AS ACTIONABLE_USD
FROM OFFICEHOURS_20261027.SAP_FINANCIALS.V_CALL_LIST;

## Finance tightens the policy
One `SET` with a qualified name changes the value for everyone. The view is untouched. Rerun the Streamlit app and the call-list tiles change too.

In [ ]:
%%sql -r set_90
SET OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE = 90;

In [ ]:
%%sql -r call_list_90
SELECT $OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE AS CUTOFF_DAYS,
       COUNT(*) AS CUSTOMERS, ROUND(SUM(ACTIONABLE_USD)) AS ACTIONABLE_USD
FROM OFFICEHOURS_20261027.SAP_FINANCIALS.V_CALL_LIST;

## Session override (what-if)
An unqualified `SET name = value` creates a **session** variable that shadows the schema variable for unqualified `$name` reads in *my* session only. The qualified read and everyone else still see 90.

In [ ]:
%%sql -r shadow
USE SCHEMA OFFICEHOURS_20261027.SAP_FINANCIALS;
SET MIN_DAYS_PAST_DUE = 60;
SELECT $MIN_DAYS_PAST_DUE AS MY_SESSION_VALUE,
       $OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE AS SCHEMA_VALUE;

In [ ]:
%%sql -r unset_shadow
UNSET MIN_DAYS_PAST_DUE;
SELECT $MIN_DAYS_PAST_DUE AS NOW_RESOLVES_TO_SCHEMA;

## Governance
READ and WRITE are separate privileges, so analysts can read the policy while only the finance role can change it.
```sql
GRANT READ  ON VARIABLE SAP_FINANCIALS.MIN_DAYS_PAST_DUE TO ROLE COLLECTIONS_ANALYST;
GRANT WRITE ON VARIABLE SAP_FINANCIALS.MIN_DAYS_PAST_DUE TO ROLE FINANCE_POLICY_ADMIN;
```

In [ ]:
DESCRIBE VARIABLE OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE;

In [ ]:
%%sql -r reset_var
SET OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE = 30;
SELECT $OFFICEHOURS_20261027.SAP_FINANCIALS.MIN_DAYS_PAST_DUE AS CUTOFF_DAYS;